# RealLift Demonstration

RealLift is a Causal Inference library for Lift Measurement, specifically designed for Geo-experiments.

This notebook demonstrates how to:
1. Install the library.
2. Generate synthetic data.
3. Run a GeoLift experiment pipeline.
4. Interpret the results.

## 1. Installation

In [ ]:
# Note: In Google Colab, you can install directly from GitHub if the repo is public
# %pip install git+https://github.com/RobertoJuniorWXYZ/RealLift.git

# For now, since we are in a local environment or testing, we install from the current directory
%pip install .

## 2. Generate Synthetic Data

We will simulate data for 10 geographical areas over 6 months.

In [ ]:
import pandas as pd
import numpy as np
from reallift import generate_geolift_data
import matplotlib.pyplot as plt

# Parameters for simulation
n_geos = 10
treatment_start_date = "2023-05-01"
treatment_geos = ["geo_0"]
true_lift = 0.15  # 15% lift

# Generate data
df, df_pre, treated = generate_geolift_data(
    start_date="2023-01-01",
    end_date="2023-07-01",
    n_geos=n_geos, 
    treatment_geos=treatment_geos, 
    treatment_start=treatment_start_date,
    lift=true_lift,
    plot=True
)

df.to_csv("simulated_data.csv", index=False)
print(f"Data simulated and saved to simulated_data.csv")
df.head()

## 3. Run GeoLift Experiment

The `run_geo_experiment` pipeline handles:
- Finding the best control groups.
- Validating the pre-treatment parallel trends.
- Running Synthetic Control.
- Performing Placebo tests.

In [ ]:
from reallift import run_geo_experiment

results = run_geo_experiment(
    filepath="simulated_data.csv",
    date_col="date",
    treatment_start_date=treatment_start_date,
    n_treatment=1,
    mde=0.1,
    plot=True,
    verbose=True
)

## 4. Summary of Results

We can inspect the summary of the experiment.

In [ ]:
print("Best Split Found:", results['summary']['splits'])

for res in results['results']:
    print(f"\nCluster Analysis: {res['cluster']}")
    print(f"Estimated Lift (abs): {res['synthetic']['lift_mean_abs']:.2f}")
    print(f"Estimated Lift (%): {res['synthetic']['lift_mean_pct']*100:.2f}%")
    print(f"Bootstrap p-value: {res['synthetic']['bootstrap']['p_value_boot']:.4f}")

## 5. Result Interpretation

To decide if your experiment was a success, consider these three indicators:

1. **Estimated Lift**: Is the lift positive and close to your expectations? In this demo, we simulated 15%.
2. **Bootstrap P-value**: A p-value < 0.05 (or 0.1 depending on your rigor) suggests the lift is statistically significant under bootstrap resampling.
3. **Placebo Test P-value**: This is a "sanity check". It treats control areas as if they were treated. 
    - If **Placebo P-value is high** (e.g., > 0.05), it means the observed lift in your true treatment group is higher than what we see in random samples of control groups. This increases confidence.
    - If **Placebo P-value is low**, it means random control groups also showed high lift, which might indicate that your "success" is just random noise or a global trend.

In [ ]:

print("=== FINAL INTERPRETATION ===")
for res in results['results']:
    p_placebo = res['placebo']['p_value']
    p_boot = res['synthetic']['bootstrap']['p_value_boot']
    lift_pct = res['synthetic']['lift_mean_pct'] * 100
    
    print(f"\nCluster: {res['cluster']['treatment']}")
    print(f"Lift: {lift_pct:.2f}%")
    print(f"P-value (Bootstrap): {p_boot:.4f}")
    print(f"P-value (Placebo): {p_placebo:.4f}")
    
    if p_boot < 0.1 and p_placebo > 0.05:
        print("✅ SUCCESS: Significant lift and passed placebo tests.")
    elif p_boot < 0.1:
        print("⚠️ CAUTION: Significant bootstrap lift, but failed/weak placebo check.")
    else:
        print("❌ FAILED: Lift is not statistically significant.")
